### **Chapter 9.6: DeePC on a Nonlinear Plant**

Chapters 9.1-9.5 used the flat Mountain Car, for which Willems' fundamental lemma holds exactly, and Chapter 9.5 deployed *flat-data* controllers on a bumpy plant. This notebook asks the more honest data-driven question:

> **We do not know the model. We can only collect data from the real (nonlinear) plant. How far does DeePC get?**

We use the bumpy terrain (`case = 3`),

$$
h(p) = k\cos(18p), \qquad \theta(p) = \arctan h'(p), \qquad \dot v = u\cos\theta - g\sin\theta\cos\theta,
$$

with a bump period of $2\pi/18 \approx 0.35\,\mathrm{m}$. Along the task from $p=-0.5$ to $p=0.6$ the car crosses three bumps, and the terrain force changes sign several times. Four questions structure the notebook:

1. What does the nonlinearity look like from the point of view of an LTI model? (*multiple equilibria, input bound*)
2. How do we even collect useful data on a plant that is not flat? (*closed-loop excitation*)
3. Where does DeePC work (*local regulation*) and where does it fail (*the transition task*)?
4. What do the regularization strength $\lambda_g$ and the history length $T_{\mathrm{ini}}$ do when the LTI assumption is violated? (*nonlinearity as noise, bias*)

As in the rest of the chapter, DeePC measures **only the position**, $y = p$. On a nonlinear plant this matters more than on the flat one: the hidden velocity has to be recovered from a short window of position samples, and the nonlinearity enters that window like measurement noise.

Case 4 (the underactuated valley) is discussed at the end; we do **not** expect an LTI behavioral model to solve it.

In [ ]:
import sys
import os
import io
import contextlib
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.env import *
from ex5_MPC.mpc_utils import LinearMPCController
from ex9_DeePC.deepc_utils import *

### **Part 1: The Nonlinear Plant Seen Through LTI Glasses**

Two facts about the bumpy terrain matter for everything that follows.

**(a) Input bound.** The terrain force $g\sin\theta\cos\theta$ with $\tan\theta = 18k$ has to be overcome by the input. With $|u| \le 1$ it saturates at $k \approx 0.0057$; above that no controller can climb a bump from rest. We use $k = 0.0025$ (peak terrain force $\approx 0.44$), so the task is comfortably feasible.

**(b) Many equilibria.** The equilibrium input $u_{\mathrm{eq}}(p) = g\sin\theta(p)\cos\theta(p)$ depends on the position. Every bump valley is a *stable* equilibrium with $u_{\mathrm{eq}} = 0$, every peak an unstable one, and a target on a slope (our $p = 0.6$) needs a non-zero holding input. An LTI model has exactly **one** equilibrium; whatever it is fitted to, it cannot know about the neighbouring valleys.

In [ ]:
freq = 20
dt = 1.0 / freq
N = 20
T_ini = 8          # longer than the lag (2): see Example 4.2
t_terminal = 6.0

bump = 0.0025
initial_state = np.array([-0.5, 0.0])
target_state = np.array([0.6, 0.0])
state_lbs = np.array([-2.0, -4.0])
state_ubs = np.array([2.0, 4.0])
input_lbs, input_ubs = -1.0, 1.0

env_plant = Env(3, initial_state, target_state, param=bump,
                state_lbs=state_lbs, state_ubs=state_ubs, input_lbs=input_lbs, input_ubs=input_ubs)
dynamics_plant = Dynamics(env_plant)

# The flat model is what the model-based baseline believes in.
env_flat = Env(1, initial_state, target_state,
               state_lbs=state_lbs, state_ubs=state_ubs, input_lbs=input_lbs, input_ubs=input_ubs)
dynamics_flat = Dynamics(env_flat)

# Cost: position error and input. DeePC only sees y = p, so its weights are scalar;
# the model-based controllers use the same cost written on the state (no velocity weight).
Q_y = np.array([[1.0]])
Q = np.diag([1.0, 0.0])
R = np.array([[0.1]])

# Terminal cost: LQR cost-to-go of the flat model (Chapter 5). With Qf = Q even the
# nonlinear MPC below prefers to park in a valley: over a 1 s horizon, staying is
# cheaper than paying R u^2 to climb. DeePC gets the position entry of the same matrix.
A_d, B_d = dynamics_flat.get_linearized_AB_discrete(target_state, np.zeros(1), dt)
Qf = scipy.linalg.solve_discrete_are(A_d, B_d, Q, R)
Qf_y = Qf[:1, :1]
K_lqr = np.linalg.solve(R + B_d.T @ Qf @ B_d, B_d.T @ Qf @ A_d)

g = 9.81
theta_max = np.arctan(18 * bump)
print(f"bump period {2*np.pi/18:.3f} m, peak terrain force {g*np.sin(theta_max)*np.cos(theta_max):.3f} (input bound {input_ubs})")
print(f"holding input at the target p=0.6: u_eq = {float(dynamics_plant.get_equilibrium_input(target_state)):.3f}")

In [ ]:
p_grid = np.linspace(-0.7, 1.2, 600)
h_vals = np.array([float(env_plant.h(p)) for p in p_grid])
u_eq_vals = np.array([float(dynamics_plant.get_equilibrium_input(np.array([p, 0.0]))) for p in p_grid])

valleys = (2 * np.arange(-2, 7) + 1) * np.pi / 18   # cos(18p) = -1
peaks = 2 * np.arange(-2, 7) * np.pi / 18           # cos(18p) = +1

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(p_grid, h_vals)
for v in valleys: ax[0].axvline(v, color="tab:green", alpha=0.3)
for pk in peaks: ax[0].axvline(pk, color="tab:red", alpha=0.3)
ax[0].axvline(initial_state[0], color="k", linestyle="--", label="start")
ax[0].axvline(target_state[0], color="tab:orange", linestyle=":", label="target")
ax[0].set_ylabel("h(p)")
ax[0].legend(loc="upper right")
ax[0].set_title("Bumpy terrain: valleys (green) are stable equilibria, peaks (red) unstable ones")

ax[1].plot(p_grid, u_eq_vals)
ax[1].axhline(input_ubs, color="gray", linestyle="--", label="input bound")
ax[1].axhline(-input_ubs, color="gray", linestyle="--")
ax[1].set_ylabel(r"$u_{eq}(p)$")
ax[1].set_xlabel("p")
ax[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

### **Part 2: Collecting Data on a Plant That Is Not Flat**

`collect_deepc_data` from Chapter 9.1 applies $u_{\mathrm{eq}}(p_{\mathrm{target}}) + e_k$ **open-loop**. On the flat plant $u_{\mathrm{eq}} = 0$ and the car performs a bounded random walk. On the bumpy plant the constant offset is the holding input *of the target slope only*; everywhere else it is wrong, the car rolls off and accelerates down the periodic terrain. The data then describe a completely different operating regime.

The standard remedy is **closed-loop identification**: a simple stabilizing feedback keeps the car in the region of interest while random excitation on top keeps the input persistently exciting,

$$
u_k = u_{\mathrm{eq}} - K\,(x_k - x_{\mathrm{target}}) + e_k,\qquad e_k \sim \mathcal U(-a, a).
$$

`collect_deepc_data_closed_loop` implements this (the feedback uses the simulator state; the *recorded* output is the position only). The price is that the data are **local**: they describe the behavior near the target, nothing else.

In [ ]:
u_open, y_open = collect_deepc_data(env_plant, dynamics_plant, freq=freq, n_samples=400,
                                    excitation_amplitude=0.8, initial_state=target_state, seed=1, output_indices=[0])
u_local, y_local = collect_deepc_data_closed_loop(env_plant, dynamics_plant, freq=freq, n_samples=600,
                                                  feedback_gain=K_lqr, excitation_amplitude=0.6, seed=1, output_indices=[0])

t_open = np.arange(y_open.shape[0]) * dt
t_local = np.arange(y_local.shape[0]) * dt
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(t_open, y_open[:, 0])
ax[0].axhspan(initial_state[0], target_state[0], color="tab:orange", alpha=0.15, label="task region")
ax[0].set_title("Open-loop excitation on the bumpy plant: the car rolls away")
ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("p"); ax[0].legend()
ax[1].plot(t_local, y_local[:, 0])
ax[1].axhspan(initial_state[0], target_state[0], color="tab:orange", alpha=0.15, label="task region")
ax[1].set_title("Closed-loop excitation: local data around the target")
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("p"); ax[1].legend()
plt.tight_layout()
plt.show()

print(f"open-loop data:   p in [{y_open[:,0].min():.2f}, {y_open[:,0].max():.2f}]")
print(f"closed-loop data: p in [{y_local[:,0].min():.3f}, {y_local[:,0].max():.3f}]")
print(f"persistently exciting (order T_ini+N+n = {T_ini+N+2}): {check_persistent_excitation(u_local - dynamics_plant.get_equilibrium_input(target_state), T_ini+N+2)}")

# Rank of the stacked Hankel matrix. For an exact LTI system of order n it is m*L + n;
# the nonlinear terrain acts like unmodelled noise and makes the matrix full row rank.
from ex9_DeePC.deepc_utils import _block_hankel
L = T_ini + N
H_local = np.vstack([_block_hankel(u_local - dynamics_plant.get_equilibrium_input(target_state), L),
                     _block_hankel(y_local[:u_local.shape[0]] - target_state[0], L)])
sv = np.linalg.svd(H_local, compute_uv=False)
print(f"stacked Hankel matrix: {H_local.shape}, LTI rank would be m*L+n = {L+2}, numerical rank = {np.linalg.matrix_rank(H_local)}")
print(f"singular values sigma_{L+2} / sigma_1 = {sv[L+1]/sv[0]:.1e}, sigma_{L+3} / sigma_1 = {sv[L+2]/sv[0]:.1e} (only a gentle decay instead of a gap: the data are not LTI data)")

### **Part 3: Where DeePC Works and Where It Fails**

We compare, with identical $Q, R, Q_f, N$ and constraints:

- **NMPC oracle**: nonlinear MPC with the *true* model (`NonlinearMPCOracle`, CasADi/ipopt). It shows what is achievable; neither of the other two sees the model.
- **Linear MPC**: Chapter-5 MPC with the flat model.
- **DeePC**: built only from the local closed-loop position data, in the regularized form of Chapter 9.5 ($\lambda_y = 10^{3}$, soft output history). The Hankel rank printed in Part 2 is the reason we do not use the deterministic (hard output-history) form: the nonlinear terrain makes the Hankel matrix full row rank, exactly as measurement noise does, so the equality constraints can match *any* history and an unregularized $g$ carries no information about the plant. Nonlinearity has to be treated like noise, and $\lambda_g$ is what turns the data into a predictor.

Two choices differ from the flat-terrain notebooks and are examined in Part 4:

- **$\lambda_g$ is scaled with the data.** $\|g\|^2$ shrinks when the data amplitude grows (the same trajectory needs smaller coefficients), so a fixed $\lambda_g$ means a different regularization for every dataset. We set $\lambda_g = \bar\lambda_g \cdot E$, where $E$ is the total energy of the deviation data, and use $\bar\lambda_g = 0.1$ - much stronger than the $10^{-3}$ of Chapter 9.5.
- **$T_{\mathrm{ini}} = 8$ instead of the lag 2.** With position-only output the velocity is a difference of noisy positions; a longer window averages the "nonlinearity noise".

The runner below records QP failures (the regularized QP stays feasible, the counter is a sanity check) and applies the equilibrium input as a fallback in that step.

In [ ]:
def run_on_plant(controller, plant, env_start, t_end=t_terminal, warmup_input=None):
    # Closed loop on `plant`. DeePC controllers are initialized with a *measured*
    # history: the first T_ini steps apply a constant warm-up input (default: the
    # target's equilibrium input) and the recorded I/O pairs seed the past window.
    # A synthetic history (repeat the current output) is not a plant trajectory.
    x = env_start.init_state.copy()
    states, inputs, failures = [x.copy()], [], 0
    u_fallback = np.atleast_1d(dynamics_plant.get_equilibrium_input(target_state))
    k0 = 0
    if isinstance(controller, DeePCController):
        u_w = u_fallback if warmup_input is None else np.atleast_1d(warmup_input)
        u_hist, y_hist = [], []
        for _ in range(controller.T_ini):
            u_hist.append(u_w.copy()); y_hist.append(controller._measure_output(x))
            x = plant.one_step_forward(x, u_w, dt)
            states.append(x.copy()); inputs.append(u_w.copy())
        controller.initialize_history(x, u_history=np.asarray(u_hist), y_history=np.asarray(y_hist))
        k0 = controller.T_ini
    for k in range(k0, int(freq * t_end)):
        try:
            with contextlib.redirect_stdout(io.StringIO()):   # silence the input-clipping notices
                u = controller.compute_action(x, k)
            u = u[0] if isinstance(u, tuple) else u
        except RuntimeError:
            failures += 1
            u = u_fallback
            if isinstance(controller, DeePCController):
                controller.initialize_history(x, mode='equilibrium')
        u = np.clip(np.asarray(u, dtype=float).reshape(-1), input_lbs, input_ubs)
        x = plant.one_step_forward(x, u, dt)
        states.append(x.copy()); inputs.append(u.copy())
    return np.asarray(states), np.asarray(inputs), failures

def closed_loop_cost(states, inputs):
    # The cost all controllers are judged by: position error, input effort, terminal position.
    dp = states[:, 0] - target_state[0]
    du = inputs[:, 0] - float(dynamics_plant.get_equilibrium_input(target_state))
    J = np.sum(Q_y[0, 0] * dp[:-1]**2 + R[0, 0] * du**2)
    return float(J + Qf_y[0, 0] * dp[-1]**2)

def data_energy(u_data, y_data, u_eq, y_ref):
    # Sum of squared deviation samples: the natural scale of ||g||^2 penalties.
    return float(np.sum((u_data - u_eq)**2) + np.sum((y_data - y_ref)**2))

def make_deepc(lambda_g_rel=0.1, lambda_y=1e3, u_data=u_local, y_data=y_local, T_ini=T_ini,
               env=env_plant, dynamics=dynamics_plant):
    u_eq = np.atleast_1d(dynamics.get_equilibrium_input(env.target_state))
    lambda_g = lambda_g_rel * data_energy(u_data, y_data, u_eq, env.target_state[0])
    return DeePCController(env, dynamics, u_data, y_data, Q_y, R, Qf_y, freq, N,
                           T_ini=T_ini, lambda_g=lambda_g, lambda_y=lambda_y, output_indices=[0],
                           history_initialization='equilibrium', name='DeePC', verbose=False)

controllers = {
    "NMPC oracle (true model)": NonlinearMPCOracle(env_plant, dynamics_plant, Q, R, Qf, freq, 2 * N),
    "Linear MPC (flat model)": LinearMPCController(env_flat, dynamics_flat, Q, R, Qf, freq, N, name='LMPC', verbose=False),
    "DeePC (regularized)": make_deepc(),
}
print(f"DeePC: T_ini={T_ini}, lambda_g = 0.1 * E = {controllers['DeePC (regularized)'].lambda_g:.2f}, lambda_y = 1e3")

#### **Example 3.1: Local regulation**

Start 5 cm below the target. This is the regime the data were collected in.

In [ ]:
env_near = Env(3, np.array([0.55, 0.0]), target_state, param=bump,
               state_lbs=state_lbs, state_ubs=state_ubs, input_lbs=input_lbs, input_ubs=input_ubs)

print(f"{'controller':<28} {'cost':>8} {'final p':>9} {'fallbacks':>10}")
for name, ctrl in controllers.items():
    if isinstance(ctrl, NonlinearMPCOracle):
        ctrl._previous = None
    X, U, fails = run_on_plant(ctrl, dynamics_plant, env_near, t_end=3.0)
    print(f"{name:<28} {closed_loop_cost(X, U):8.3f} {X[-1,0]:9.3f} {fails:10d}")

In the regime the data come from, **DeePC regulates as well as the nonlinear oracle** (cost 0.05 vs 0.04) - from position measurements alone: the local data span the local behavior, and the fundamental lemma "almost" holds. The flat linear MPC, although it is handed the full state, leaves an offset because it does not know the holding input $u_{\mathrm{eq}}(0.6) \ne 0$.

#### **Example 3.2: The transition task**

Now start from $p = -0.5$, three bumps away.

In [ ]:
results = {}
for name, ctrl in controllers.items():
    if isinstance(ctrl, NonlinearMPCOracle):
        ctrl._previous = None
    results[name] = run_on_plant(ctrl, dynamics_plant, env_plant)

print(f"{'controller':<28} {'cost':>8} {'final p':>9} {'fallbacks':>10}")
for name, (X, U, fails) in results.items():
    print(f"{name:<28} {closed_loop_cost(X, U):8.3f} {X[-1,0]:9.3f} {fails:10d}")

In [ ]:
t_x = np.arange(int(freq * t_terminal) + 1) * dt
t_u = np.arange(int(freq * t_terminal)) * dt
fig, ax = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for name, (X, U, _) in results.items():
    ax[0].plot(t_x, X[:, 0], label=name)
    ax[1].plot(t_x, X[:, 1], label=name)
    ax[2].plot(t_u, U[:, 0], label=name)
for v in valleys:
    if -0.6 < v < 1.2: ax[0].axhline(v, color="tab:green", alpha=0.25)
ax[0].axhline(target_state[0], color="tab:orange", linestyle=":", label="target")
ax[0].set_ylabel("position (valleys in green)"); ax[0].legend(loc="lower right", fontsize=8)
ax[1].set_ylabel("velocity")
ax[2].set_ylabel("input"); ax[2].set_xlabel("time (s)")
fig.suptitle(f"Transition on the bumpy plant, k={bump}: NMPC vs flat linear MPC vs local-data DeePC")
plt.tight_layout()
plt.show()

#### **Results Analysis**

- The **nonlinear oracle** reaches the target: it knows that the terrain force changes sign along the way.
- The **flat linear MPC** stalls in the valley *before* the target (around $p \approx 0.52$-$0.54$): it has no gravity term at all, and a proportional-like controller without integral action settles where its model error is balanced.
- **DeePC** is feasible at every step and drives the car across the bumps, but it overshoots into the valley *after* the target ($p \approx 0.87$). Its behavioral model is a linearization *including the holding input* $u_{\mathrm{eq}}(0.6)$; applied at other positions this constant offset is wrong, and once the car is past the target the next valley is the nearest place where the controller's model and the plant agree.

The two LTI-based controllers therefore fail in the same structural way - **they land one bump period away from the target**, in opposite directions decided by which constant input offset they assume. The cost numbers tell the story only partially: the flat MPC's cost is within 4 % of the oracle's (stopping 5 cm short is cheap), while DeePC pays about 50 % more for travelling one bump too far and coming to rest on the wrong slope. A terminal-position check is more telling than the cost here.

### **Part 4: Regularization and History Length Outside the LTI Regime**

Chapter 9.5 introduced $\lambda_y$ (soft output-history matching) and $\lambda_g$ (coefficient regularization) as a robustification. On the nonlinear plant they are not optional, and their values matter. Three sweeps, all with the amplitude-0.6 dataset from Part 2, and one extrapolation experiment.

#### **Example 4.1: The regularization strength $\lambda_g$**

We vary $\bar\lambda_g$ (so $\lambda_g = \bar\lambda_g E$) over six orders of magnitude and run both tasks.

In [ ]:
def evaluate(ctrl_factory, label_fmt, values):
    rows = []
    for v in values:
        Xn, Un, _ = run_on_plant(ctrl_factory(v), dynamics_plant, env_near, t_end=3.0)
        Xt, Ut, ft = run_on_plant(ctrl_factory(v), dynamics_plant, env_plant)
        rows.append((v, closed_loop_cost(Xn, Un), Xn[-1, 0], closed_loop_cost(Xt, Ut), Xt[-1, 0], ft))
        print(label_fmt.format(v) + f" | local: cost {rows[-1][1]:7.3f} final p {rows[-1][2]:6.3f} | transition: cost {rows[-1][3]:7.2f} final p {rows[-1][4]:6.3f} fallbacks {ft}")
    return rows

def sweep_plot(rows, xlabel, log_x=True):
    xs = [r[0] for r in rows]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot = ax[0].semilogx if log_x else ax[0].plot
    plot(xs, [r[4] for r in rows], marker="o", label="transition (from -0.5)")
    plot(xs, [r[2] for r in rows], marker="s", label="local regulation (from 0.55)")
    ax[0].axhline(target_state[0], color="tab:orange", linestyle=":", label="target")
    for v in valleys:
        if -0.2 < v < 1.4: ax[0].axhline(v, color="tab:green", alpha=0.25)
    ax[0].set_ylim(-0.3, 1.5); ax[0].set_xlabel(xlabel); ax[0].set_ylabel("final position (valleys in green)"); ax[0].legend()
    plot = ax[1].semilogx if log_x else ax[1].plot
    plot(xs, [r[3] for r in rows], marker="o", label="transition")
    ax[1].axhline(closed_loop_cost(*results["NMPC oracle (true model)"][:2]), color="k", linestyle="--", label="NMPC oracle")
    ax[1].set_yscale("log"); ax[1].set_xlabel(xlabel); ax[1].set_ylabel("closed-loop cost (transition)"); ax[1].legend()
    plt.tight_layout(); plt.show()

lambda_rel_values = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
rows_lambda = evaluate(lambda v: make_deepc(lambda_g_rel=v), "lambda_g_rel {:7.0e}", lambda_rel_values)
sweep_plot(rows_lambda, r"$\bar\lambda_g$")

- **Too little regularization** ($\bar\lambda_g \le 10^{-3}$, i.e. $\lambda_g \lesssim 0.1$ - the range that works perfectly on the flat plant in Chapter 9.5): the controller does not even regulate from 5 cm below the target, and from the far start it runs away. The Hankel matrix is full rank, the cheap directions of $g$ are dominated by the nonlinearity, and the "predictor" is noise.
- **A sweet spot** around $\bar\lambda_g \approx 10^{-1}$: local regulation matches the oracle and the transition lands one valley late (at $10^{-2}$ the local behavior is already fine, but the transition overshoots by two valleys). Regularization has turned the nonlinear data into a *local linear model*.
- **Too much regularization** ($\bar\lambda_g \ge 1$): the predictions are pulled towards $g = 0$, i.e. towards "nothing happens"; local regulation degrades and the transition becomes erratic. (That $\bar\lambda_g = 1$ happens to stop near the target is a coincidence of bias and terrain, not knowledge: the same controller no longer regulates locally.)

#### **Example 4.2: The history length $T_{\mathrm{ini}}$**

On the flat plant, $T_{\mathrm{ini}} = 2$ (the lag) is enough to pin down the hidden velocity from positions (Chapter 9.4). Here the positions are "noisy".

In [ ]:
T_ini_values = [2, 3, 4, 6, 8, 12, 16]
rows_Tini = evaluate(lambda v: make_deepc(T_ini=v), "T_ini {:3d}", T_ini_values)
sweep_plot(rows_Tini, r"$T_{\mathrm{ini}}$", log_x=False)

With $T_{\mathrm{ini}} = 2$ the implicit velocity estimate is a single finite difference of two positions that do not exactly obey any LTI model; the controller overshoots badly even locally. From $T_{\mathrm{ini}} \approx 4$-$6$ on, the window is long enough to average the mismatch, and the results are stable up to $T_{\mathrm{ini}} = 16$ (where the cost slowly grows again: 0.8 s of past nonlinear behavior now has to be explained by one LTI model). Position-only DeePC on a nonlinear plant therefore needs a **longer history than the lag** - the same effect noisy measurements have.

#### **Example 4.3: Excitation amplitude**

Finally the amount of terrain the data cover. Thanks to the data-scaled $\lambda_g$, the result is nearly independent of the amplitude: whether the data cover a few millimetres or several tens of centimetres around the target, the local linear model that comes out lands the car in the valley after the target. Wider data average more slopes into the model, which costs a little accuracy but does not change the conclusion.

In [ ]:
amplitudes = [0.05, 0.1, 0.2, 0.3, 0.6, 1.2, 2.0]
print(f"{'amplitude':>9} {'data p-range':>18} {'cost':>8} {'final p':>8}")
for amp in amplitudes:
    u_a, y_a = collect_deepc_data_closed_loop(env_plant, dynamics_plant, freq=freq, n_samples=600,
                                              feedback_gain=K_lqr, excitation_amplitude=amp, seed=1, output_indices=[0])
    Xa, Ua, _ = run_on_plant(make_deepc(u_data=u_a, y_data=y_a), dynamics_plant, env_plant)
    print(f"{amp:9.2f}   [{y_a[:,0].min():+.3f}, {y_a[:,0].max():+.3f}] {closed_loop_cost(Xa, Ua):8.2f} {Xa[-1,0]:8.3f}")

#### **Example 4.4: Regularization bias when extrapolating**

Now place the target in a bump valley ($p = \pi/6 \approx 0.524$, $u_{\mathrm{eq}} = 0$, a *stable* equilibrium). The valley is stiff, so the closed-loop data cover only a few centimetres. From the far start the controller has to extrapolate 1 m away from its data - and we can look at what it *predicts* before it moves.

In [ ]:
valley_target = np.array([np.pi / 6, 0.0])
env_valley = Env(3, initial_state, valley_target, param=bump,
                 state_lbs=state_lbs, state_ubs=state_ubs, input_lbs=input_lbs, input_ubs=input_ubs)
dynamics_valley = Dynamics(env_valley)
A_v, B_v = dynamics_flat.get_linearized_AB_discrete(valley_target, np.zeros(1), dt)
K_v = np.linalg.solve(R + B_v.T @ Qf @ B_v, B_v.T @ Qf @ A_v)
u_valley, y_valley = collect_deepc_data_closed_loop(env_valley, dynamics_valley, freq=freq, n_samples=600,
                                                    feedback_gain=K_v, excitation_amplitude=0.6, seed=1, output_indices=[0])
print(f"valley data: p in [{y_valley[:,0].min():.3f}, {y_valley[:,0].max():.3f}]")

make_valley_deepc = lambda: make_deepc(u_data=u_valley, y_data=y_valley, env=env_valley, dynamics=dynamics_valley)

# First prediction after a measured warm-up (T_ini steps with u = u_eq = 0, standing on the slope at -0.5).
ctrl = make_valley_deepc()
x = env_valley.init_state.copy()
u_hist, y_hist = [], []
for _ in range(T_ini):
    u_hist.append(np.zeros(1)); y_hist.append(x[[0]])
    x = dynamics_valley.one_step_forward(x, np.zeros(1), dt)
ctrl.initialize_history(x, u_history=np.asarray(u_hist), y_history=np.asarray(y_hist))
with contextlib.redirect_stdout(io.StringIO()):
    u, y_pred, _ = ctrl.compute_action(x, T_ini)
print(f"car at p={x[0]:+.3f}, v={x[1]:+.3f}: first input {float(u[0]):+.3f}, predicted next position {float(y_pred[1,0]):+.3f}, predicted position after N steps {float(y_pred[-1,0]):+.3f}")

X, U, fails = run_on_plant(make_valley_deepc(), dynamics_valley, env_valley, t_end=4.0)
print(f"final p {X[-1,0]:+.3f} (target {valley_target[0]:.3f}), fallbacks {fails}")

t_v = np.arange(X.shape[0]) * dt
plt.figure(figsize=(10, 3.5))
plt.plot(t_v, X[:, 0], label="DeePC")
for v in valleys:
    if -7.5 < v < 1.0: plt.axhline(v, color="tab:green", alpha=0.25)
plt.axhline(valley_target[0], color="tab:orange", linestyle=":", label="target (valley)")
plt.xlabel("time (s)"); plt.ylabel("position (valleys in green)"); plt.legend()
plt.title("Valley target: biased predictions far from the data")
plt.tight_layout(); plt.show()

The very first prediction is already wrong by 20 cm: standing at $p \approx -0.51$ and creeping downhill, the controller predicts the next position at $-0.31$ and believes it will be at $p \approx 1.3$, far *past* the target, after one second. With data that cover a few centimetres, matching the true history 1 m away would require a coefficient $g$ of enormous norm, and $\lambda_g\|g\|^2$ wins over $\lambda_y\|Y_{\mathrm{ini}}g - y_{\mathrm{meas}}\|^2$: the prediction is pulled towards the data. Acting on it, the controller brakes at full authority, the car rolls backwards down the periodic terrain and never comes back.

Regularization therefore does not "handle" nonlinearity. It makes the problem well posed at the price of **bias** (predictions pulled towards the data) - and the bias grows exactly when the data are least representative, i.e. far from the region they were collected in. Close to the data (Example 3.1) the bias is harmless; when the controller has to reason about a region it has never seen, it is not.

### **Part 5: Case 4 and the Boundary of the Method**

The underactuated valley (`case = 4`, $h = \sin 3p$) is not attempted here. Solving it requires *exploiting* the nonlinearity - swinging back to gain energy before climbing - which no single LTI behavior can represent: every trajectory of an LTI system is a linear combination of the data columns, and "swing back to go forward" is not in the span of small-excitation data collected anywhere in the valley. Chapters 5 (nonlinear MPC), 4 (iLQR) and 7/8 (RL) are the tools for that regime.

What *does* extend DeePC beyond a local linear model - all active research topics:

- **online / adaptive DeePC**: refresh the Hankel matrix from the most recent closed-loop data so that the local model follows the operating point (needs continued excitation and careful windowing);
- **lifting**: apply DeePC to Koopman or kernel features of the outputs, so that the nonlinear behavior becomes (approximately) linear in the lifted space;
- **nonlinear fundamental lemmas** for specific system classes (e.g. Hammerstein/Wiener, flat systems, polynomial systems).

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway: On a nonlinear plant, DeePC is a data-driven local linear MPC**

On the bumpy Mountain Car, regularized DeePC built from closed-loop position data around the target regulates as well as the nonlinear oracle - but across the bumps it fails exactly like the flat linear MPC, landing one valley past the target: one equilibrium, one constant input offset. Everything that distinguishes the nonlinear from the flat case behaves like measurement noise: the Hankel matrix becomes full rank, $\lambda_g$ must be orders of magnitude larger (and scaled with the data), the position-only history must be longer than the lag, and far from the data the regularization biases the predictions instead of making the plant linear. Case 4, which needs the nonlinearity to be *exploited*, is outside what any single LTI behavior can express.
</blockquote>

**References:** Coulson, Lygeros, and Dörfler, *Data-Enabled Predictive Control: In the Shallows of the DeePC*, ECC 2019; Berberich, Köhler, Müller, and Allgöwer, *Data-Driven Model Predictive Control With Stability and Robustness Guarantees*, IEEE TAC 2021 (regularized DeePC, tracking with online data); Lian, Wang, and Jones, *Koopman based data-driven predictive control*, 2021; Markovsky and Dörfler, *Behavioral systems theory in data-driven analysis, signal processing, and control*, Annual Reviews in Control 2021.